In [1]:
import os
from dotenv import load_dotenv
from google import genai
import pandas as pd 
import nbformat 



In [2]:
load_dotenv() # loading my env file
notebook_path = "/Users/ahmadwali04/Desktop/LeetCode/code.ipynb"
client = genai.Client()


def load_notebook_to_df(notebook_path):
    try:
        with open(notebook_path, 'r', encoding='utf-8') as f:
            nb=nbformat.read(f,as_version=4)
            rows = []
            for cell in nb.cells:
                if cell.cell_type in ['markdown','code']:
                    rows.append({
                        "chunk type": cell.cell_type,
                        "chunk" : cell.source.strip()
                    })
            df = pd.DataFrame(rows)
            return df
    except FileNotFoundError:
        print("File not found")
        return None 
    
def ask_about_chunk(df,index,question):
    selected_row = df.loc[index]
    chunk_type = selected_row["chunk type"]
    chunk_content = selected_row["chunk"]

    prompt = f"""
    You are an expert software engineer and LeetCode mentor. 
    Below is a specific content chunk extracted from my LeetCode Jupyter Notebook (Index: {index}).

    --- CHUNK TYPE: {chunk_type.upper()} ---
    {chunk_content}
    ---------------------------------

    USER QUESTION ABOUT THIS CHUNK:
    {question}
    """

    print(f"Sending Chunk #{index} to Gemini...")
        
    try:
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt
        )
        
        print("\n=== GEMINI'S ANALYSIS ===")
        print(response.text)
        print("=========================")
        
    except Exception as e:
        print(f"An error occurred while calling the Gemini API: {e}")


df = load_notebook_to_df(notebook_path)


# Display the dataframe (this will show the index, chunk type, and snippet)
if df is not None:
    # Set display options so pandas doesn't truncate the text too aggressively
    pd.set_option('display.max_colwidth', 100)
    print("Notebook parsed successfully! Here are your chunks:")
    display(df)

Notebook parsed successfully! Here are your chunks:


,chunk type,chunk
0,markdown,"# Contains Duplicate\nGiven an integer array nums, return true if any value appears more than on..."
1,code,"class Solution:\n def hasDuplicate(self, nums: List[int]) -> bool:\n x = set(nums)\n ..."
2,markdown,"# Valid Anagram\nGiven two strings s and t, return true if the two strings are anagrams of each ..."
3,code,"class Solution:\n def isAnagram(self, s: str, t: str) -> bool:\n if len(s) != len(t):\..."
4,markdown,"# Two Sum\n\nGiven an array of integers nums and an integer target, return the indices i and j s..."
5,code,"class Solution:\n def twoSum(self, nums: List[int], target: int) -> List[int]:\n seen ..."
6,markdown,
7,markdown,"# Group Anagrams\nGiven an array of strings strs, group all anagrams together into sublists. You..."
8,markdown,


In [14]:
my_question = "Explain what this problem is and how a solution might look"
ask_about_chunk(df, index=3, question=my_question)

Sending Chunk #3 to Gemini...

=== GEMINI'S ANALYSIS ===
As an expert software engineer and LeetCode mentor, let me break this down for you.

### 1. What is this problem?
This is the classic **"Valid Anagram"** problem (LeetCode #242).

**Definition:** An anagram is a word or phrase formed by rearranging the letters of a different word or phrase, typically using all the original letters exactly once.

**The core logic:** Two strings are anagrams if and only if they contain the **exact same characters with the exact same frequencies**.
*   *Example:* `"listen"` and `"silent"` are anagrams.
*   *Example:* `"rat"` and `"car"` are not anagrams.

---

### 2. How the solution works
Your provided code uses a **Frequency Counter (Hash Map)** approach. Here is the step-by-step breakdown of the logic:

#### Step A: Early Exit (Edge Case)
```python
if len(s) != len(t):
    return False
```
This is a constant-time $O(1)$ check. If the lengths differ, they cannot possibly be anagrams. Always look f

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def load_notebook_to_df_st(path):
    with open(path, "r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)

    rows = []
    for cell_index, cell in enumerate(nb.cells):
        if cell.cell_type in ["markdown", "code"]:
            rows.append(
                {
                    "cell_index": cell_index,
                    "chunk type": cell.cell_type,
                    "chunk": cell.source.strip(),
                }
            )

    return pd.DataFrame(rows)


def build_embeddings_st(df):
    chunks = df["chunk"].fillna("").tolist()
    return st_model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )


def retrieve_chunks_st(df, embeddings, question, top_k=3):
    question_embedding = st_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

    scores = np.dot(embeddings, question_embedding)
    top_indices = scores.argsort()[::-1][:top_k]
    results = df.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    return results


def ask_rag_question_st(question, top_k=3):
    local_df = load_notebook_to_df_st(notebook_path)
    embeddings = build_embeddings_st(local_df)
    retrieved = retrieve_chunks_st(local_df, embeddings, question, top_k=top_k)

    context = "\n\n".join(
        f"[cell {row.cell_index}] ({row['chunk type']}, score={row.score:.3f})\n{row.chunk}"
        for row in retrieved.itertuples(index=False)
    )

    prompt = f"""
You are an expert software engineer and LeetCode mentor.
Use only the notebook context below to answer the question.
If the context is insufficient, say what is missing and what you would need next.

NOTEBOOK CONTEXT:
{context}

USER QUESTION:
{question}
"""

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
    )

    print(response.text)
    return retrieved


question = "Explain what this notebook chunk is doing and suggest a better approach."
retrieved = ask_rag_question_st(question, top_k=3)
display(retrieved)

/Users/ahmadwali04/Desktop/personal/Projects/NoteGraph/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 23462.41it/s]



TypeError: tuple indices must be integers or slices, not str

In [ ]:
def ask_rag_question_st_fixed(question, top_k=3):
    local_df = load_notebook_to_df_st(notebook_path)
    embeddings = build_embeddings_st(local_df)
    retrieved = retrieve_chunks_st(local_df, embeddings, question, top_k=top_k)

    context_parts = []
    for _, row in retrieved.iterrows():
        context_parts.append(
            f"[cell {row['cell_index']}] ({row['chunk type']}, score={row['score']:.3f})\n{row['chunk']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are an expert software engineer and LeetCode mentor.
Use only the notebook context below to answer the question.
If the context is insufficient, say what is missing and what you would need next.

NOTEBOOK CONTEXT:
{context}

USER QUESTION:
{question}
"""

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
    )

    print(response.text)
    return retrieved


question = "Explain what this notebook chunk is doing and suggest a better approach."
retrieved = ask_rag_question_st_fixed(question, top_k=3)
display(retrieved)